<a href="https://colab.research.google.com/github/Arif0000/llm-app/blob/main/SIMPLE_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -U langchain-google-genai langchain-chroma langchain-core

import os
os.environ["GOOGLE_API_KEY"] = "enter you API keys"

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_chroma import Chroma
from langchain_core.documents import Document


# Create Documents

docs = [
    Document(
        page_content="Virat Kohli is one of the most successful batsmen in IPL history...",
        metadata={"team": "Royal Challengers Bangalore"}
    ),
    Document(
        page_content="Rohit Sharma is the most successful IPL captain...",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="MS Dhoni is known as Captain Cool...",
        metadata={"team": "Chennai Super Kings"}
    ),
    Document(
        page_content="Jasprit Bumrah is one of the best fast bowlers...",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="Ravindra Jadeja is a top all-rounder...",
        metadata={"team": "Chennai Super Kings"}
    )
]


#Embeddings (REQUIRED)

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)


# Vector DB

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="my_chroma_db",
    collection_name="sample"
)

vector_store.add_documents(docs)


# Gemini 2.5 Flash LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)


# Similarity Search

retrieved_docs = vector_store.similarity_search(
    query="Who are the bowlers?",
    k=2
)

for doc in retrieved_docs:
    print(doc.page_content)


# RAG (LLM + Retrieval)

context = "\n".join([doc.page_content for doc in retrieved_docs])

query = "Who are the bowlers?"

prompt = f"""
Answer the question based on the context below:

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print("\nGemini Answer:\n", response.content)

Jasprit Bumrah is one of the best fast bowlers...
Ravindra Jadeja is a top all-rounder...

Gemini Answer:
 Based on the context:

*   **Jasprit Bumrah** (explicitly stated as a "fast bowler")
*   **Ravindra Jadeja** (as a "top all-rounder," he is also a bowler)
